In [0]:
from typing import Any

In [0]:
import logging
from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = "../../helpers/lng_lnd"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = "../../helpers/curated_helpers"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_cadastre_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # -------SCHEMA
        # schema = (
        #     "CadastreId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "CadastreKey string,"
        #     "RealEstateKey string,"
        #     "CepalKey string,"
        #     "CooperatorKey string,"
        #     "DateApplied timestamp,"
        #     "DateUpdated timestamp,"
        #     "DateCompleted timestamp,"
        #     "StatusCode string,"
        #     "KaekNo string,"
        #     "KaekOffice string,"
        #     "RegStage string,"
        #     "XtraFld1 string,"
        #     "XtraFld2 string,"
        #     "Comments string,"
        #     "EnteredBy int,"
        #     "DateEntered timestamp,"
        #     "ActiveProperty boolean,"
        #     "PropertyStatus string,"
        #     "RegistryCheckDate timestamp,"
        #     "CadastreStatus string,"
        #     "CadastreSubcategory string,"
        #     "CadastralOfficeOta string,"
        #     "OtaCode string,"
        #     "StartDate timestamp,"
        #     "EndDate timestamp,"
        #     "Priority boolean,"
        #     "FolderComplete string,"
        #     "CompleteComments string,"
        #     "CorrectData string,"
        #     "CorrectDataComments string,"
        #     "SuggestedAction string,"
        #     "BusinessDate  date"
        # )

        cluster_columns = ["BusinessDate"]

        # -------CHOOSE PATH

        curated_table_path = f"{curated_zone}.Cadastre"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "thesis",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # --------APPLY FILTERS AND SELECT COLUMNS FROM THE TABLES WE NEED

        # ================== THESIS_PRCADASTRE ==================

        cadastre_extra_filters = None

        if cadastre_extra_filters is not None:
            cadastre_extra_filters.append(date_filter)
            cadastre_final_filters = " AND ".join(cadastre_extra_filters)
        else:
            cadastre_final_filters = date_filter

        cadastre_selected_fields = [
            "id",
            "portfolio_code",
            "property_id",
            "cepal_key",
            "cooperator_id",
            "date_applied",
            "date_updated",
            "date_completed",
            "status_code",
            "kaek_no",
            "kaek_office",
            "reg_stage",
            "xtra_fld1",
            "xtra_fld2",
            "comments",
            "entered_by",
            "date_entered",
            "active_property",
            "property_status",
            "registry_check_date",
            "cadastre_status",
            "cadastre_subcat",
            "cadastral_office_ota",
            "ota_code",
            "date_start",
            "date_end",
            "priority",
            "folder_complete",
            "complete_comments",
            "correct_data",
            "correct_data_comments",
            "suggested_action",
            "BusinessDate",
        ]

        CADASTRE = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prcadastre",
            cadastre_final_filters,
            cadastre_selected_fields,
        )

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "PortfolioKey",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # -------SET THE CONDITIONS FOR JOINS
        PrCadastre_Portfolio_join_condition = (
            CADASTRE["portfolio_code"] == portfolio["Prefix"]
        )

        # -------APPLY THE JOINS
        cadastre_df = CADASTRE.join(
            portfolio,
            PrCadastre_Portfolio_join_condition,
            "left",
        )

        # --------SET THE KEYS FOR HASHKEY BASED ON DATA MAPPING
        CadastreId_primary_key_fields = [
            CADASTRE.id,
            lit("thesis"),
        ]

        # -------CREATION OF HΑSKEY
        cadastre_df = cadastre_df.withColumn(
            "CadastreId",
            sha2(
                concat_ws(";", *CadastreId_primary_key_fields),
                256,
            ),
        )

        # -------SELECTION OF FINAL COLUMNS
        result_df_thesis = cadastre_df.select(
            cadastre_df.CadastreId.alias("CadastreId"),
            lit("thesis").alias("SourceSystem"),
            portfolio.PortfolioKey.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit("None").alias("BatchId"),
            CADASTRE.id.cast("string").alias("CadastreKey"),
            CADASTRE.property_id.cast("string").alias("RealEstateKey"),
            CADASTRE.cepal_key.cast("string").alias("CepalKey"),
            CADASTRE.cooperator_id.cast("string").alias("CooperatorKey"),
            CADASTRE.date_applied.alias("DateApplied"),
            CADASTRE.date_updated.alias("DateUpdated"),
            CADASTRE.date_completed.alias("DateCompleted"),
            CADASTRE.status_code.alias("StatusCode"),
            CADASTRE.kaek_no.alias("KaekNo"),
            CADASTRE.kaek_office.alias("KaekOffice"),
            CADASTRE.reg_stage.alias("RegStage"),
            CADASTRE.xtra_fld1.alias("XtraFld1"),
            CADASTRE.xtra_fld2.alias("XtraFld2"),
            CADASTRE.comments.alias("Comments"),
            CADASTRE.entered_by.alias("EnteredBy"),
            CADASTRE.date_entered.alias("DateEntered"),
            CADASTRE.active_property.alias("ActiveProperty"),
            CADASTRE.property_status.alias("PropertyStatus"),
            CADASTRE.registry_check_date.alias("RegistryCheckDate"),
            CADASTRE.cadastre_status.alias("CadastreStatus"),
            CADASTRE.cadastre_subcat.alias("CadastreSubcategory"),
            CADASTRE.cadastral_office_ota.alias("CadastralOfficeOta"),
            CADASTRE.ota_code.alias("OtaCode"),
            CADASTRE.date_start.alias("StartDate"),
            CADASTRE.date_end.alias("EndDate"),
            CADASTRE.priority.alias("Priority"),
            CADASTRE.folder_complete.alias("FolderComplete"),
            CADASTRE.complete_comments.alias("CompleteComments"),
            CADASTRE.correct_data.alias("CorrectData"),
            CADASTRE.correct_data_comments.alias("CorrectDataComments"),
            CADASTRE.suggested_action.alias("SuggestedAction"),
            CADASTRE.BusinessDate.alias("BusinessDate"),
        )
        # --------------APPEND DATA TO CURATED
        result_df_thesis.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_cadastre_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)